# Benchmark

Evaluate each model with **gen_quality + TailGAN-style VaR/ES** (no our-way eval_portfolio).

Generation order per model (only **ours** can regenerate/train in this pipeline):
1. if `synthetic_data/<tag>/` already has npy -> skip generation, evaluate only;
2. else if a weight (`quantDiffusion_best*.pt`) exists -> load it and regenerate;
3. else -> train from scratch, then generate.

- **ours** = `synthetic_data/baseline/` (Close_to_Close/w10 full model = RV decomp 12d + LambertW + DMS-TCN, small_size=5).
- baseline models (QuantGAN, TailGAN, Diffusion-TS) are external architectures -> only step 1 applies (skip if npy missing).
- summary tables: `result/statistics/benchmark__gen_quality.csv`, `benchmark__tailgan_vares.csv`.

In [1]:
import os, sys, glob, copy
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from utils.utils import *
from utils.metric import *
from utils.file_manager import LOCAL_BASE_DIRECTORY, quantDiffusionPath
from models.DMS_TCN import DMSTCN
from torch.utils.data import DataLoader, TensorDataset
from modules.diffusion import TimeSeriesGaussianDiffusion, generate_cosine_schedule

file_manager = quantDiffusionPath()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEQ_LEN, N_RUNS = 256, 5
SD = os.path.join(LOCAL_BASE_DIRECTORY, "result", "statistics", "synthetic_data")

# ----- ours config (matches the ablation baseline weight) -----
RV_NAME, RV_WINDOW = "Close_to_Close", 10
LARGE_SIZE, SMALL_SIZE = [15, 7, 5], [5, 5, 5]
NUM_EPOCHS, VAL_START, CLIP_LATENT, TARGET_N = 45, 25, 8.0, 2000
OURS_TAG = "baseline"

# models: label -> (tag/folder, regenerable). Only ours uses our diffusion pipeline.
MODELS = {
    "QuantDiffusion (ours, C2C/w10)": ("baseline", True),
    "QuantGAN": ("quantgan", False),
    "TailGAN": ("tailgan", False),
    "Diffusion-TS": ("diffusionTS", False),
}

# ----- data: real (test 6d), train (val ref + bounds) -----
df = pd.read_csv("./data/indices.csv")
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
df.set_index("Date", inplace=True)
df = df.apply(pd.to_numeric).astype(float)
df_log_returns = np.log1p(df.pct_change()).dropna()
train_size = int(0.7 * len(df_log_returns))
train_raw = df_log_returns.iloc[:train_size, :]
test_raw = df_log_returns.iloc[train_size:, :]
real_data = torch.from_numpy(rolling_window(test_raw.values, SEQ_LEN)).float().transpose(1, 2).cpu().numpy()
train_ref = torch.from_numpy(rolling_window(train_raw.values, SEQ_LEN)).float().transpose(1, 2).cpu().numpy()
mn = train_ref.min(axis=(0, 2), keepdims=True); mx = train_ref.max(axis=(0, 2), keepdims=True)
rng = (mx - mn) * 0.1
lower_bound, upper_bound = mn - rng, mx + rng
print("real_data (test 6d):", real_data.shape)

# ----- TailGAN-style evaluator -----
_TG_DIR = os.path.join(LOCAL_BASE_DIRECTORY, "modules", "TailGAN")
if _TG_DIR not in sys.path:
    sys.path.insert(0, _TG_DIR)

# build the TailGAN rolling-window CSVs (data/TailGAN/gan_data/indices2_6d) if missing (needed by the metric)
import run as _tg_run   # import-safe: run.py pipeline is __main__-guarded
_gan_dir = os.path.join(_tg_run.YOUR_PATH, "gan_data", _tg_run.DATA_NAME)
if glob.glob(os.path.join(_gan_dir, "*.csv")):
    print("[TailGAN] gan_data exists -> skip build_windows()")
else:
    print("[TailGAN] building rolling-window CSVs -> " + _gan_dir)
    _tg_run.build_windows()

import tailgan_eval
_TG = tailgan_eval.configure_from_project(LOCAL_BASE_DIRECTORY)
TG_ALPHAS = (0.05, 0.025, 0.01)


def _pm(m, s, nd=4):
    return f"{m:.{nd}f} ± {s:.{nd}f}" if (np.isfinite(m) and np.isfinite(s)) else ""

c:\Users\Juhwankim\miniconda3\envs\research\lib\site-packages\pandas\core\internals\blocks.py:395: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)


real_data (test 6d): (1538, 6, 256)
[TailGAN] gan_data exists -> skip build_windows()
Namespace(n_epochs=3000, batch_size=1000, lr_D=1e-07, lr_G=1e-06, temp=0.01, b1=0.5, b2=0.999, latent_dim=1000, len=50000, n_rows=5, n_cols=100, n_critic_G=1, n_critic_D=1, static_way='LShort', strategies=['Port', 'MR', 'TF'], n_trans=50, Cap=10, WH=10, ratios=[1.0, 1.0], thresholds_pct=[[31, 69]], data_name='1_Gauss+1_AR50+1_AR-12+1_GARCH-T5+1_GARCH-T10', tickers=['Gauss', 'AR50', 'AR-12', 'GARCH-T5', 'GARCH-T10'], noise_name='t5', alphas=[0.05], W=10.0, score='quant', numNN=10, project=True, version='Test1')


In [2]:
# ===== ours (baseline) generation/training pipeline =====
# scalers matching the ablation baseline (C2C/w10 + RV floor + full scaling incl. LambertW)
def _floor_rv(_rv):
    _rv = _rv.copy()
    for _c in _rv.columns:
        _pos = _rv[_c][_rv[_c] > 0]
        _rv[_c] = _rv[_c].clip(lower=float(_pos.quantile(0.01)) if len(_pos) else 1e-8)
    return _rv


def _fit_scalers(rv_window=RV_WINDOW):
    eps = 1e-8
    rv = _floor_rv(pd.read_csv(file_manager.get_rv_file(RV_NAME), index_col=0).rename_axis("Date"))
    rv.index = pd.to_datetime(rv.index)
    ci = df_log_returns.index.intersection(rv.index)
    dlr, drv = df_log_returns.loc[ci], rv.loc[ci]
    frv = np.sqrt(drv.rolling(rv_window, min_periods=min(5, rv_window)).mean().shift(-(rv_window - 1)))
    combined = pd.concat([dlr / (frv + eps), np.log(frv + eps)], axis=1).dropna()
    scaled_12d, scalers = scaling(combined.values)
    return scaled_12d, scalers


_SCALED12, SCALERS = _fit_scalers()
TRAIN_12D = _SCALED12[:train_size]


def _to6d(gen_12d):
    inv = inverse_scaling(gen_12d.transpose(1, 2), SCALERS)
    return inv[:, :6, :] * np.exp(np.clip(inv[:, 6:, :], -20.0, 10.0))


def _build_ours():
    ts = DMSTCN({"enc_in": 12, "dims": [128, 128, 128], "large_size": LARGE_SIZE,
                 "small_size": SMALL_SIZE, "time_emb_dim": 128, "use_time_emb": True,
                 "use_cross_attention": False, "max_seq_len": SEQ_LEN,
                 "use_sequence_pos_encoding": False})
    betas = generate_cosine_schedule(75, 1.5, 1e-5, 0.5)
    return TimeSeriesGaussianDiffusion(
        ts, seq_len=SEQ_LEN, num_channels=12, betas=betas, loss_type="11",
        ema_decay=0.9999, ema_start=2000, ema_update_rate=1,
        sampling_type="student_t", prediction_type="epsilon").to(device)


def _train_ours(out_dir):
    """tier-3: train the full ours model, save best weight to out_dir."""
    os.makedirs(out_dir, exist_ok=True)
    dl = DataLoader(TensorDataset(torch.from_numpy(rolling_window(TRAIN_12D, SEQ_LEN)).float()),
                    batch_size=256, shuffle=False)
    diffusion = _build_ours()
    opt = torch.optim.Adam(diffusion.parameters(), lr=2e-4, weight_decay=1e-5)
    best_state, best_emd, best_ep = None, float("inf"), -1
    for epoch in tqdm(range(1, NUM_EPOCHS + 1), desc="[ours] train", leave=False):
        diffusion.train()
        for (x,) in dl:
            x = x.to(device)
            loss = diffusion(x)
            opt.zero_grad(); loss.backward(); opt.step(); diffusion.update_ema()
            del x, loss
        if epoch >= VAL_START:
            diffusion.eval()
            try:
                with torch.no_grad():
                    fk = _to6d(torch.clamp(diffusion.sample(batch_size=100, device=device, use_ema=True),
                                           -CLIP_LATENT, CLIP_LATENT))
                    emd = compute_avg_emd(train_ref, fk, 100)
                if np.isfinite(emd) and emd < best_emd:
                    best_emd, best_ep, best_state = emd, epoch, copy.deepcopy(diffusion.state_dict())
            except Exception:
                pass
        clean_memory(device)
    if best_state is not None:
        diffusion.load_state_dict(best_state)
        torch.save(diffusion.state_dict(), os.path.join(out_dir, f"quantDiffusion_best({best_ep}).pt"))
        print(f"  [ours] trained, best epoch {best_ep} (EMD {best_emd:.5f})")
    return diffusion


def _regen_ours(out_dir, diffusion=None):
    """tier-2/3 generation: survivor-filtered sampling -> save N_RUNS npy."""
    os.makedirs(out_dir, exist_ok=True)
    if diffusion is None:
        best = max(glob.glob(f"{out_dir}/quantDiffusion_best*.pt"), key=os.path.getmtime)
        diffusion = _build_ours()
        diffusion.load_state_dict(torch.load(best, map_location=device))
        print(f"  [ours] loaded {os.path.basename(best)}")
    diffusion.eval()

    def _surv(raw):
        s = torch.clamp(torch.cat(raw, dim=0), -CLIP_LATENT, CLIP_LATENT)
        with np.errstate(over="ignore", invalid="ignore"):
            f6 = _to6d(s)
        m = np.isfinite(f6).all(axis=(1, 2)) & np.all((f6 >= lower_bound) & (f6 <= upper_bound), axis=(1, 2))
        return f6[m]

    for run_idx in range(N_RUNS):
        raw = []
        with torch.no_grad():
            for _ in range(int(np.ceil(TARGET_N / 0.6 / 100))):
                raw.append(diffusion.sample(batch_size=100, device=device, use_ema=True))
        surv = _surv(raw)
        while len(surv) < TARGET_N and len(raw) * 100 < 2600:
            rate = max(len(surv), 1) / (len(raw) * 100)
            with torch.no_grad():
                for _ in range(int(np.ceil((TARGET_N - len(surv)) / max(rate, 0.2) / 100)) + 1):
                    raw.append(diffusion.sample(batch_size=100, device=device, use_ema=True))
            surv = _surv(raw)
        clean_memory(device)
        final = surv[np.random.choice(len(surv), TARGET_N, replace=False)] if len(surv) >= TARGET_N else surv
        np.save(os.path.join(out_dir, f"{run_idx}.npy"), final)
        print(f"  [ours] run {run_idx + 1}/{N_RUNS}: {final.shape}")


def ensure_ours(out_dir):
    """3-tier: existing npy > weight -> regenerate > train -> generate."""
    if len(glob.glob(f"{out_dir}/[0-9]*.npy")) >= N_RUNS:
        print("  [ours] tier-1: existing npy -> skip generation"); return
    if glob.glob(f"{out_dir}/quantDiffusion_best*.pt"):
        print("  [ours] tier-2: weight found -> regenerate"); _regen_ours(out_dir); return
    print("  [ours] tier-3: no npy/weight -> train then generate")
    diffusion = _train_ours(out_dir)
    _regen_ours(out_dir, diffusion)

In [3]:
# ===== evaluation (gen_quality + TailGAN VaR/ES) =====
def eval_gen_quality(npy_dir, n_runs=N_RUNS):
    windows = [1, 5, 20, 100]
    total = pd.DataFrame()
    for seed in range(n_runs):
        f = os.path.join(npy_dir, f"{seed}.npy")
        if not os.path.exists(f):
            continue
        fake = np.load(f)
        rdf = pd.DataFrame()
        rdf = pd.concat([rdf, calculate_ccf(real_data, fake, 0)], axis=0)
        ccf = pd.DataFrame(columns=["Average"])
        for i in range(1, 11):
            ccf = pd.concat([ccf, calculate_ccf(real_data, fake, i)], axis=1)
        rdf = pd.concat([rdf, pd.DataFrame(ccf.mean(axis=1), columns=["Average"])], axis=0)
        rdf = pd.concat([rdf, calculate_distribution_scores(real_data, fake, windows)], axis=0)
        rdf = pd.concat([rdf, calculate_skew_kurtosis(real_data, fake, windows)], axis=0)
        _, acf_df = calculate_acf_score(real_data, fake, 40)
        rdf = pd.concat([rdf, acf_df], axis=0)
        _, lev_df = calculate_leverage_effect(real_data, fake, 40)
        rdf = pd.concat([rdf, lev_df], axis=0)
        total = pd.concat([total, rdf], axis=1)
    stat = pd.DataFrame(index=total.index, columns=["MEAN", "STD"])
    stat["MEAN"] = total.mean(axis=1).values
    stat["STD"] = total.std(axis=1).values
    return stat


def eval_tailgan(npy_dir, n_runs=N_RUNS):
    per_seed = []
    for seed in range(n_runs):
        f = os.path.join(npy_dir, f"{seed}.npy")
        if not os.path.exists(f):
            continue
        fake = np.load(f).astype(np.float32)
        per_seed.append(tailgan_eval.vares_relerror(_TG.Compute_PNL, _TG.Tensor, real_data, fake, TG_ALPHAS))
    keys = list(per_seed[0].keys())
    arr = np.array([[d[k] for k in keys] for d in per_seed])
    return keys, arr.mean(0), arr.std(0)

# Run: ensure synthetic (per order above) + evaluate + summary tables

In [4]:
gq_cols, tg_cols = {}, {}
for label, (tag, regenerable) in MODELS.items():
    d = os.path.join(SD, tag)
    print(f"\n===== {label} ({tag}) =====")
    if regenerable:
        ensure_ours(d)  # tier1 npy > tier2 weight > tier3 train (ours only)
    n = len(glob.glob(os.path.join(d, "[0-9]*.npy")))
    if n == 0:
        print(f"  (skip {label}: no npy at {d}; external model cannot be regenerated here)")
        continue
    print(f"  evaluate ({n} npy)")
    stat = eval_gen_quality(d)
    gq_cols[label] = pd.Series([_pm(m, s) for m, s in zip(stat["MEAN"], stat["STD"])], index=stat.index)
    keys, m, s = eval_tailgan(d)
    tg_cols[label] = pd.Series([_pm(a, b) for a, b in zip(m, s)], index=keys)

gq_summary = pd.DataFrame(gq_cols)
tg_summary = pd.DataFrame(tg_cols)
gq_summary.to_csv(file_manager.get_result_file("benchmark__gen_quality"), encoding="utf-8-sig")
tg_summary.to_csv(file_manager.get_result_file("benchmark__tailgan_vares"), encoding="utf-8-sig")
print("\nsaved: benchmark__gen_quality.csv, benchmark__tailgan_vares.csv")
print("\n[Benchmark - TailGAN-style VaR/ES]")
print(tg_summary.to_string())
print("\n[Benchmark - Generation Quality]")
print(gq_summary.to_string())


===== QuantDiffusion (ours, C2C/w10) (baseline) =====
  [ours] tier-1: existing npy -> skip generation
  evaluate (5 npy)

===== QuantGAN (quantgan) =====
  evaluate (5 npy)

===== TailGAN (tailgan) =====
  evaluate (5 npy)

===== Diffusion-TS (diffusionTS) =====
  evaluate (5 npy)

saved: benchmark__gen_quality.csv, benchmark__tailgan_vares.csv

[Benchmark - TailGAN-style VaR/ES]
              QuantDiffusion (ours, C2C/w10)         QuantGAN          TailGAN     Diffusion-TS
RE_VaR(0.05)                 0.3815 ± 0.0062  0.4468 ± 0.0102  0.4604 ± 0.0101  0.6333 ± 0.0045
RE_ES(0.05)                  0.3507 ± 0.0040  0.5126 ± 0.0157  0.4020 ± 0.0056  0.5195 ± 0.0169
RE_VaR(0.025)                0.3525 ± 0.0082  0.4925 ± 0.0133  0.4320 ± 0.0058  0.5434 ± 0.0186
RE_ES(0.025)                 0.3455 ± 0.0029  0.5693 ± 0.0225  0.3895 ± 0.0045  0.4670 ± 0.0215
RE_VaR(0.01)                 0.3566 ± 0.0032  0.5412 ± 0.0122  0.3899 ± 0.0042  0.4790 ± 0.0221
RE_ES(0.01)                  0.3533 ± 0